# 🪔 Akshara-Drishti — MobileNetV2 Dual-Pipeline Training (96x96)
**AI-Powered Indic Script Intelligence System**

Classifies 6 inscriptions: `brahmi`, `devanagari`, `kannada`, `malayalam`, `tamil`, `telugu`

---
> ⚠️ **Run Cell 1 first, then restart the runtime before continuing.**

## 🛠️ Step 0: Dataset Configuration

**To improve model accuracy and handle the domain gap:**
1. Ensure you have ~800 Train / 150 Val / 150 Test images per class.
2. Synthetics generated via Google Noto Sans fonts.
3. Include the `BrahmiGAN` Zenodo extract in `dataset/brahmi/`.
4. Real data finetuning (Few-shot) handles domain gap.

In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================
!pip install -q tensorflow==2.15.0 opencv-python-headless pillow matplotlib seaborn scikit-learn

In [ ]:
# ============================================================
# CELL 2 — ALL IMPORTS
# ============================================================
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from PIL import Image

print(f"✅ TensorFlow version: {tf.__version__}")

In [ ]:
# ============================================================
# CELL 3 — PATHS & HYPERPARAMETERS
# ============================================================
BASE_PATH    = "/content/drive/MyDrive/AKSHARA_DHRISTI_SOFTSKILLS"
DATASET_PATH = BASE_PATH + "/dataset"
FINAL_PATH   = BASE_PATH + "/final_model.keras"

IMG_SIZE   = 96
BATCH_SIZE = 32
EPOCHS_S1  = 12   # Stage 1 — frozen base, lr=1e-3
EPOCHS_S2  = 15   # Stage 2 — unfreeze top 40 layers, lr=1e-5
SEED       = 42
ALLOWED_CLASSES = ["brahmi", "devanagari", "kannada", "malayalam", "tamil", "telugu"]

In [ ]:
# ============================================================
# CELL 4 — LOAD DATASET & CLAHE PIPELINE
# ============================================================
# Load data
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=ALLOWED_CLASSES, label_mode='categorical'
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=ALLOWED_CLASSES, label_mode='categorical'
)

# Basic scaling
train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
val_ds   = val_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))

train_ds = train_ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
# ============================================================
# CELL 5 — DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1), # shift
    layers.RandomBrightness(0.2),
    layers.RandomFlip("horizontal"),
], name="augmentation")

In [ ]:
# ============================================================
# CELL 6 — BUILD MOBILENETV2 (96x96)
# ============================================================
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # frozen for Stage 1

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = data_augmentation(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(ALLOWED_CLASSES), activation='softmax')(x)

model = Model(inputs, outputs, name="AksharaDrishti-MobileNetV2")
model.summary()

In [ ]:
# ============================================================
# CELL 7 — STAGE 1: TRAIN (Frozen Base)
# ============================================================
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
callbacks = [early_stop, reduce_lr]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_S1, callbacks=callbacks)

In [ ]:
# ============================================================
# CELL 8 — STAGE 2: FINE-TUNE (Top 40 layers)
# ============================================================
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_S2, callbacks=callbacks)

In [ ]:
# ============================================================
# CELL 9 — EVALUATION & SAVE
# ============================================================
print("Evaluating on Validation Set...")
model.save(FINAL_PATH)
print(f"\n✅ Final model saved to: {FINAL_PATH}")